<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks/4.3-training-building-a-cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Implementación de Red Neuronal Convolusional (CNN) mediante el uso de tensores de PyTorch

Este notebook entrena una CNN desde cero usando los tensores 3D generados en el notebook 3.2.
También incluye un modelo tradicional (Random Forest) con grid search para comparar desempeño.


### 1. Imports y configuración base


In [11]:
# Imports
#------------------------------------------------------------------------------------------
import os
import copy
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd
import time
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from google.colab import drive
from tqdm import tqdm

In [2]:
# Semilla y runtime
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


### 2. Carga de tensores (train/val/test)

Los archivos `.pt` generados en el notebook 3.2 contienen un diccionario con `x`, `y`,
metadatos y el mapeo de clases. Ajusta `BASE_DIR_TENSOR` según tu ruta local/Drive.


In [3]:
drive.mount('/content/drive')
! cp -r /content/drive/MyDrive/split_pytorch_tensors /content/split_pytorch_tensors

Mounted at /content/drive


In [4]:
BASE_DIR_TENSOR = '/content/split_pytorch_tensors'
BATCH_SIZE = 64

SPLIT_FILES = {
    'train': 'train_tensors.pt',
    'val': 'val_tensors.pt',
    'test': 'test_tensors.pt',
}

def load_pack(split_name: str):
    path = os.path.join(BASE_DIR_TENSOR, SPLIT_FILES[split_name])
    if not os.path.exists(path):
        raise FileNotFoundError(f'No existe el archivo: {path}')
    return torch.load(path, map_location='cpu', weights_only=False)

# Cargamos lo packs en variables aun no son tensores puros
train_pack = load_pack('train')
val_pack = load_pack('val')
test_pack = load_pack('test')

# Verificamos las clases del diccionario dentro del pack
class_to_idx = train_pack['class_to_idx']
idx_to_class = {v: k for k, v in class_to_idx.items()}
class_names = [idx_to_class[i] for i in sorted(idx_to_class.keys())]
print('Clases:', class_names)
print('Shape train:', tuple(train_pack['x'].shape))

Clases: ['angry', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
Shape train: (7422, 3, 60, 51)


In [7]:
class TensorPackDataset(Dataset):
    def __init__(self, pack):
        self.x = pack['x'].float() # [N, 3, n_mels, targetframes]
        self.y = pack['y'].long()  # len(N)

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [5]:
class EarlyStopping:
  def __init__(self, patience=4, min_delta=0.0):
    self.patience = patience
    self.min_delta = min_delta
    self.best = None
    self.counter = 0

  def step(self, metric):
    if self.best is None or metric > self.best + self.min_delta:

      self.best = metric
      self.counter = 0
      return False

    self.counter += 1
    return self.counter >= self.patience

In [8]:
def build_dataloaders(batch_size=BATCH_SIZE):
    pin = torch.cuda.is_available()
    train_ds = TensorPackDataset(train_pack)
    val_ds = TensorPackDataset(val_pack)
    test_ds = TensorPackDataset(test_pack)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=pin)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=pin)

    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = build_dataloaders()


### 3. Definición de la CNN


In [9]:
class EmotionCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Caracterisdticas de los datos  (Filtrado)
        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),

            nn.BatchNorm2d(32), # Normalizar valores de la convolusion, para seprar mas la fuente de la emoción.
            nn.ReLU(), # Necesaria para evitar hacer muy lineal la Red
            nn.MaxPool2d(2), # Reducimos las dimensiones para aumetar variabilidad y rendimiento.

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),

            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        # Clasificación de los datos a partir de las caracteristicas
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


### 4. Entrenamiento y validación


In [12]:
# TRAIN
#-------------------------------------------------------------------------------
def train_one_epoch(model, loader, criterion, optimizer, device, epoch=None):

    model.train()
    running_loss = 0.0
    running_correct = 0

    # Configuramos la barra de progreso (para conjunto de entrenamiento, label y unidades)
    pbar = tqdm(loader, desc=f"Epoca{epoch:02d} [train]", leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad() # Reinicia gradientes antes del paso de retropropagación
        outputs = model(inputs) # Copara in vs outs
        loss = criterion(outputs, labels) # Mide los de la predicción
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()

        # Log corto en barra
        pbar.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_correct / len(loader.dataset)
    return epoch_loss, epoch_acc

@torch.no_grad()

# VAL
#-------------------------------------------------------------------------------

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    # Configuramos la barra de progreso (para conjunto de validación, label y unidades)
    pbar = tqdm(loader, desc=f"Epoca {epoch:02d} [val]", leave=False)
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        pbar.set_postfix(loss=loss.item())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = running_correct / len(loader.dataset)
    return epoch_loss, epoch_acc

In [13]:
model = EmotionCNN(num_classes=len(class_names)).to(device)
# Cross loss por clase
criterion = nn.CrossEntropyLoss()
# Usamos Adam es el mas comun en esta clase de clasificación
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


EPOCHS = 20
early_stopper = EarlyStopping(patience=5, min_delta=0.001)

best_state = copy.deepcopy(model.state_dict())
best_val_acc = 0.0
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch=epoch)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device, epoch=epoch)

    elapsed = time.time() - start
    log_line = (
        f"[{time.strftime('%H:%M:%S')}] "
        f"Epoch {epoch:02d} | "
        f"Train loss {train_loss:.4f} acc {train_acc:.4f} | "
        f"Val loss {val_loss:.4f} acc {val_acc:.4f} | "
        f"Time {elapsed:.1f}s"
    )
    print(log_line)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "time_sec": elapsed
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    # Early stopping
    if early_stopper.step(val_acc):
        print(f"Early stopping en epoch {epoch} (best val_acc={best_val_acc:.4f})")
        break

model.load_state_dict(best_state)
print(f"Mejor accuracy en validación: {best_val_acc:.4f}")

# =========================
# Reporte final + tabla de F1 por clase
# =========================
y_true, y_pred = get_predictions(model, test_loader, device)

# Reporte completo si lo deseas
print(classification_report(y_true, y_pred, target_names=class_names))

# Tabla solo F1-score por clase
report = classification_report(
    y_true, y_pred, target_names=class_names, output_dict=True
)
f1_table = pd.DataFrame({
    "clase": class_names,
    "f1_score": [report[c]["f1-score"] for c in class_names]
})
print("\nF1-score por clase:")
display(f1_table)

KeyboardInterrupt: 

### 5. Evaluación final en test


In [ ]:
@torch.no_grad()
def get_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    for inputs, labels in loader:
        outputs = model(inputs.to(device))
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(labels.numpy())
    return np.concatenate(all_targets), np.concatenate(all_preds)

y_true, y_pred = get_predictions(model, test_loader, device)
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap='Blues', xticks_rotation=45)
plt.tight_layout()
plt.show()


## Modelo tradicional de ML con Grid Search

Se utiliza un **Random Forest** con búsqueda de hiperparámetros sobre el set de validación.


In [ ]:
def pack_to_numpy(pack):
    x = pack['x'].float().reshape(pack['x'].shape[0], -1).numpy()
    y = pack['y'].numpy()
    return x, y

X_train, y_train = pack_to_numpy(train_pack)
X_val, y_val = pack_to_numpy(val_pack)
X_test, y_test = pack_to_numpy(test_pack)

X_train_val = np.concatenate([X_train, X_val], axis=0)
y_train_val = np.concatenate([y_train, y_val], axis=0)

test_fold = np.concatenate([
    -1 * np.ones(len(X_train), dtype=int),
    np.zeros(len(X_val), dtype=int)
])
predefined_split = PredefinedSplit(test_fold)


In [ ]:
param_grid = {
    'n_estimators': [200, 400],
    'max_depth': [None, 20, 40],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
}

rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)
grid = GridSearchCV(
    rf,
    param_grid=param_grid,
    cv=predefined_split,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train_val, y_train_val)

print('Mejores parámetros:', grid.best_params_)
print('Mejor f1_macro (val):', grid.best_score_)


In [ ]:
best_rf = grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
print(classification_report(y_test, y_pred_rf, target_names=class_names))

cm_rf = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(cm_rf, display_labels=class_names).plot(cmap='Greens', xticks_rotation=45)
plt.tight_layout()
plt.show()
